# 验证门禁

agent 不能够将它自己的工作标记为完成。验证门径读范围契约、反馈日志、规则报告以及差异记录，来回答一个问题：这个任务是否真的完成了？如果门禁说没完成，任务就没做完，不管agent怎么说。

## 问题描述

agent 很容易就宣告成功，有三种主导的失败模式：
1. “看起来对的”。  模型读差异记录，然后认为是正确的。
2. “测试过痛了”。  自信地宣告，但是没有测试真正执行过的记录。
3. “达到验收”。  验收标准被宽松解读到“任何像完成的东西”都行。

工作台的修法是加一个验证门禁，读取agent产生的物料然后做出宣言。门禁是决定性的，且受版本控制的。关卡接近了CI，agent没法贿赂它。

## 基本概念

```mermaid
flowchart TD
A[Diff]-->B[verify_agent.py]
C[scope_report.json]-->B
D[rule_report.json]-->B
E[feedback_record.jsonl]-->B
B-->F[verification_report.json]
F-->G(passed?)
G--yes-->H[Reviewer Agent]
G--no-->I[Refuse done + surface to human]
```

### 关卡检查哪些内容

|项目|物料来源|
|---|---|
|所有的验收命令跑了、且返回值正常|feedback_record.jsonl|
|范围检查没有写入禁止项、或者越界|scope_report.json|
|block严重度的规则通过|rule_report.json|
|反馈里没有null退出码|feedback_record.jsonl|
|修改文件匹配allowed_files|都有|

严重等级为warn交给裁决加注。严重等级为block阻止门禁通过。

### 确定性，而非概率性

门禁每次从相同物料产出的结果必须保持一致，不包含LLM的判断。LLM判断属于审查问题，那里的目标是定性评估，而不是状态评估。

### 一份报告一个路径

门禁在每个任务结束时吐出一个`verification_report.json`，写在`outputs/verification/<task_id>.json`。CI在相同路径下消费，多个门禁时路径不同但源自同一份事实。

### 无例外地拒绝

block级别的发现不能被agent覆盖。只能被人覆盖，然后带一个记录在案的`override_reason`和`overriden_by`用户id。每次覆盖对应一次签名变更，而不是一次agent决定。

# 开始编码

对应本章核心：**验证门禁读物料做宣言（agent 不能自评完成）**、**验收命令/范围/规则 block/null exit 确定性检查**、**`outputs/verification/<task_id>.json`**、**block 只能人类签名覆盖**。  
玩具先跑通三种失败与覆盖；生产段用 **LangChain 工具 + DeepSeek** 强制经门禁才能 `claim_done`。不硬凑 PyTorch。


## 1. 教学玩具：VerificationGate

- 物料：`scope_report.json` / `rule_report.json` / `feedback_record.jsonl` / `changed_files.json`。
- warn 只加注，block 挡通过；同一物料同一 `artifact_hash`。
- `claim_done` 失败抛 `DoneRefused`；`human_override` 必须有 `overridden_by` + `override_reason`。


In [ ]:
from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Literal


Severity = Literal["block", "warn"]


@dataclass
class Finding:
    """单条确定性检查结果。"""

    check: str
    ok: bool
    severity: Severity
    detail: str

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


@dataclass
class VerificationReport:
    """门禁产物：同一物料必须得到同一结论。"""

    task_id: str
    passed: bool
    findings: list[dict[str, Any]] = field(default_factory=list)
    warn_notes: list[str] = field(default_factory=list)
    override_reason: str | None = None
    overridden_by: str | None = None
    artifact_hash: str = ""

    def to_dict(self) -> dict[str, Any]:
        return asdict(self)


class DoneRefused(RuntimeError):
    """门禁未通过：任务不能标完成。"""


def _read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def _read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    rows: list[dict[str, Any]] = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def hash_artifacts(payloads: dict[str, Any]) -> str:
    """
    Args:
        payloads: 门禁读取的全部物料。
    Returns:
        digest: 物料指纹，保证确定性可复现。
    """
    blob = json.dumps(payloads, ensure_ascii=False, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


class VerificationGate:
    """读范围/规则/反馈/差异，决定任务是否真的完成。不含 LLM。"""

    def __init__(self, root: Path) -> None:
        """
        Args:
            root: 任务工作目录。
        """
        self.root = Path(root)
        self.scope_path = self.root / "scope_report.json"
        self.rule_path = self.root / "rule_report.json"
        self.feedback_path = self.root / "feedback_record.jsonl"
        self.diff_path = self.root / "changed_files.json"
        self.contract_path = self.root / "scope_contract.json"

    def report_path(self, task_id: str) -> Path:
        """
        Args:
            task_id: 任务 id。
        Returns:
            path: outputs/verification/<task_id>.json
        """
        return self.root / "outputs" / "verification" / f"{task_id}.json"

    def evaluate(self, task_id: str) -> VerificationReport:
        """
        对同一物料确定性打分。

        Args:
            task_id: 任务 id。
        Returns:
            report: 含 findings 与 passed。
        """
        scope = _read_json(self.scope_path)
        rules = _read_json(self.rule_path)
        contract = _read_json(self.contract_path)
        feedback = _read_jsonl(self.feedback_path)
        changed = list(_read_json(self.diff_path) or [])
        if isinstance(changed, dict):
            changed = list(changed.get("files", []))

        findings: list[Finding] = []
        acceptance = list(contract.get("acceptance_criteria", []))
        allowed = list(contract.get("allowed_files", []))

        findings.append(self._check_acceptance_ran(acceptance, feedback))
        findings.append(self._check_acceptance_exit(acceptance, feedback))
        findings.append(self._check_no_null_exit(feedback))
        findings.append(self._check_scope(scope))
        findings.append(self._check_rules(rules))
        findings.append(self._check_changed_allowed(changed, allowed, scope))

        blocks = [f for f in findings if not f.ok and f.severity == "block"]
        warns = [f for f in findings if not f.ok and f.severity == "warn"]
        artifact_hash = hash_artifacts(
            {
                "scope": scope,
                "rules": rules,
                "contract": contract,
                "feedback": feedback,
                "changed": changed,
            }
        )
        report = VerificationReport(
            task_id=task_id,
            passed=not blocks,
            findings=[f.to_dict() for f in findings],
            warn_notes=[f"{f.check}: {f.detail}" for f in warns],
            artifact_hash=artifact_hash,
        )
        return report

    def write_report(self, report: VerificationReport) -> Path:
        """
        Args:
            report: 门禁报告。
        Returns:
            path: 落盘路径。
        """
        dest = self.report_path(report.task_id)
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_text(json.dumps(report.to_dict(), ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
        return dest

    def run(self, task_id: str) -> VerificationReport:
        """
        评估并写入固定路径。

        Args:
            task_id: 任务 id。
        Returns:
            report: 已落盘报告。
        """
        report = self.evaluate(task_id)
        self.write_report(report)
        return report

    def claim_done(self, task_id: str) -> VerificationReport:
        """
        Agent 不能自行标完成：必须读已有报告或重新跑门禁。

        Args:
            task_id: 任务 id。
        Returns:
            report: 通过的报告。
        Raises:
            DoneRefused: block 未通过且无人覆盖。
        """
        path = self.report_path(task_id)
        if path.exists():
            data = json.loads(path.read_text(encoding="utf-8"))
            report = VerificationReport(**data)
        else:
            report = self.run(task_id)
        if report.passed:
            return report
        if report.overridden_by and report.override_reason:
            return report
        raise DoneRefused(f"verification failed for {task_id}; refuse done")

    def human_override(self, task_id: str, *, user_id: str, reason: str) -> VerificationReport:
        """
        仅人类可覆盖 block，且必须留下签名。

        Args:
            task_id: 任务 id。
            user_id: 覆盖人。
            reason: 记录在案的原因。
        Returns:
            report: 带 override 字段的报告。
        """
        if not user_id.strip() or not reason.strip():
            raise ValueError("override requires overridden_by and override_reason")
        path = self.report_path(task_id)
        if path.exists():
            report = VerificationReport(**json.loads(path.read_text(encoding="utf-8")))
        else:
            report = self.evaluate(task_id)
        report.overridden_by = user_id
        report.override_reason = reason
        report.passed = True
        self.write_report(report)
        return report

    @staticmethod
    def _cmd_key(command: Any) -> str:
        if isinstance(command, list):
            return " ".join(str(x) for x in command)
        return str(command)

    def _check_acceptance_ran(self, acceptance: list[str], feedback: list[dict[str, Any]]) -> Finding:
        ran = {self._cmd_key(r.get("command", "")) for r in feedback}
        missing = [cmd for cmd in acceptance if cmd not in ran]
        ok = not missing
        return Finding(
            check="acceptance_ran",
            ok=ok,
            severity="block",
            detail="all acceptance commands recorded" if ok else f"missing={missing}",
        )

    def _check_acceptance_exit(self, acceptance: list[str], feedback: list[dict[str, Any]]) -> Finding:
        by_cmd: dict[str, list[dict[str, Any]]] = {}
        for row in feedback:
            by_cmd.setdefault(self._cmd_key(row.get("command", "")), []).append(row)
        bad: list[str] = []
        for cmd in acceptance:
            rows = by_cmd.get(cmd, [])
            if not rows or any(r.get("exit_code") != 0 for r in rows):
                bad.append(cmd)
        ok = not bad
        return Finding(
            check="acceptance_exit",
            ok=ok,
            severity="block",
            detail="acceptance exits are 0" if ok else f"nonzero_or_missing={bad}",
        )

    def _check_no_null_exit(self, feedback: list[dict[str, Any]]) -> Finding:
        nulls = [r.get("id", "?") for r in feedback if r.get("exit_code") is None]
        ok = not nulls
        return Finding(
            check="no_null_exit",
            ok=ok,
            severity="block",
            detail="no null exits" if ok else f"null_ids={nulls}",
        )

    def _check_scope(self, scope: dict[str, Any]) -> Finding:
        forbidden = list(scope.get("forbidden_hits") or [])
        out = list(scope.get("out_of_scope") or [])
        ok = not forbidden and not out and bool(scope.get("ok", not forbidden and not out))
        if "ok" in scope:
            ok = bool(scope["ok"]) and not forbidden and not out
        return Finding(
            check="scope_clean",
            ok=ok,
            severity="block",
            detail="scope clean" if ok else f"forbidden={forbidden} out_of_scope={out}",
        )

    def _check_rules(self, rules: dict[str, Any]) -> Finding:
        verdicts = list(rules.get("verdicts") or [])
        blocks = [v.get("rule_id", "?") for v in verdicts if v.get("severity") == "block" and not v.get("ok", True)]
        if not verdicts and "passed" in rules:
            ok = bool(rules["passed"])
            return Finding(
                check="rules_block",
                ok=ok,
                severity="block",
                detail="rules passed" if ok else "rule_report.passed=false",
            )
        ok = not blocks
        return Finding(
            check="rules_block",
            ok=ok,
            severity="block",
            detail="no block-severity rule failures" if ok else f"block_rules={blocks}",
        )

    def _check_changed_allowed(self, changed: list[str], allowed: list[str], scope: dict[str, Any]) -> Finding:
        in_scope = list(scope.get("in_scope") or [])
        extra = [p for p in changed if p not in in_scope] if in_scope else []
        # warn：改动与 allowed 的宽松提示，不挡门（allowed 是 glob，这里只核对 scope 已声明的 in_scope）
        if extra:
            return Finding(
                check="changed_match_allowed",
                ok=False,
                severity="warn",
                detail=f"changed not in scope.in_scope: {extra}",
            )
        return Finding(
            check="changed_match_allowed",
            ok=True,
            severity="warn",
            detail=f"changed={changed} allowed_globs={allowed}",
        )


def write_artifacts(
    root: Path,
    *,
    task_id: str = "T-LOGIN",
    scope_ok: bool = True,
    rules_ok: bool = True,
    acceptance_ok: bool = True,
    null_exit: bool = False,
    extra_changed: bool = False,
) -> Path:
    """
    构造门禁可读物料。

    Args:
        root: 工作目录。
        task_id: 任务 id。
        scope_ok: 范围报告是否干净。
        rules_ok: 规则是否无 block 失败。
        acceptance_ok: 验收命令是否 exit 0。
        null_exit: 是否插入 null 退出。
        extra_changed: 是否多记一个未在 in_scope 的文件（warn）。
    Returns:
        root: 同一目录。
    """
    root.mkdir(parents=True, exist_ok=True)
    contract = {
        "task_id": task_id,
        "goal": "修复登陆校验",
        "allowed_files": ["src/auth/**/*.py"],
        "forbidden_files": ["README.md"],
        "acceptance_criteria": ["python -m pytest -q tests/test_login.py"],
        "rollback_plan": "git checkout -- src/auth",
    }
    (root / "scope_contract.json").write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding="utf-8")
    scope = {
        "ok": scope_ok,
        "in_scope": ["src/auth/login.py"],
        "out_of_scope": [] if scope_ok else ["README.md"],
        "forbidden_hits": [] if scope_ok else ["README.md"],
    }
    (root / "scope_report.json").write_text(json.dumps(scope, ensure_ascii=False, indent=2), encoding="utf-8")
    rules = {
        "passed": rules_ok,
        "verdicts": [
            {"rule_id": "R-DOD-1", "ok": rules_ok, "severity": "block", "detail": "pytest"},
            {"rule_id": "R-STYLE-1", "ok": True, "severity": "warn", "detail": "lint skipped"},
        ],
    }
    (root / "rule_report.json").write_text(json.dumps(rules, ensure_ascii=False, indent=2), encoding="utf-8")
    cmd = contract["acceptance_criteria"][0]
    rows = [
        {
            "id": "fb-acc",
            "command": cmd.split(),
            "stdout_tail": "1 passed" if acceptance_ok else "1 failed",
            "stderr_tail": "",
            "exit_code": 0 if acceptance_ok else 1,
            "duration_ms": 120,
            "started_at": 1.0,
            "agent_note": "run acceptance",
            "error": None,
            "parent_id": None,
        }
    ]
    if null_exit:
        rows.append(
            {
                "id": "fb-null",
                "command": ["python", "-c", "hang"],
                "stdout_tail": "",
                "stderr_tail": "",
                "exit_code": None,
                "duration_ms": 0,
                "started_at": 2.0,
                "agent_note": "timeout",
                "error": "timeout",
                "parent_id": None,
            }
        )
    (root / "feedback_record.jsonl").write_text(
        "".join(json.dumps(r, ensure_ascii=False) + "\n" for r in rows),
        encoding="utf-8",
    )
    changed = ["src/auth/login.py"]
    if extra_changed:
        changed.append("docs/notes.md")
    (root / "changed_files.json").write_text(json.dumps(changed, ensure_ascii=False), encoding="utf-8")
    return root


print("verification gate ready | deterministic artifacts → report")


## 2. 玩具示例

绿路径、看起来对/测试过痛/null exit、warn 不挡、人类签名覆盖。


In [ ]:
def demo_verification_gate() -> None:
    """通过 / 三种失败 / 确定性复现 / agent 拒完成 / 仅人类覆盖。"""
    root = Path(tempfile.mkdtemp(prefix="verify_gate_"))
    try:
        write_artifacts(root, scope_ok=True, rules_ok=True, acceptance_ok=True)
        gate = VerificationGate(root)
        r1 = gate.run("T-LOGIN")
        assert r1.passed
        dest = gate.report_path("T-LOGIN")
        assert dest.exists()
        gate.claim_done("T-LOGIN")
        r1b = gate.evaluate("T-LOGIN")
        assert r1.artifact_hash == r1b.artifact_hash
        print("green path + deterministic hash ok")

        # 看起来对：范围越界
        write_artifacts(root, scope_ok=False)
        r_scope = gate.run("T-LOGIN")
        assert not r_scope.passed
        assert any(f["check"] == "scope_clean" and not f["ok"] for f in r_scope.findings)
        try:
            gate.claim_done("T-LOGIN")
            raise AssertionError("agent must not mark done")
        except DoneRefused:
            print("looks-right blocked by scope ok")

        # 测试过痛了：验收未绿
        write_artifacts(root, acceptance_ok=False)
        r_acc = gate.run("T-LOGIN")
        assert not r_acc.passed
        assert any(f["check"] == "acceptance_exit" and not f["ok"] for f in r_acc.findings)
        print("tests-hurt blocked ok")

        # 宽松验收：null exit
        write_artifacts(root, null_exit=True)
        r_null = gate.run("T-LOGIN")
        assert not r_null.passed
        assert any(f["check"] == "no_null_exit" and not f["ok"] for f in r_null.findings)
        print("null exit blocked ok")

        # warn 不挡通过
        write_artifacts(root, extra_changed=True)
        r_warn = gate.run("T-LOGIN")
        assert r_warn.passed
        assert r_warn.warn_notes
        print("warn annotates but does not block ok")

        # 人类覆盖：必须有签名
        write_artifacts(root, rules_ok=False)
        r_fail = gate.run("T-LOGIN")
        assert not r_fail.passed
        try:
            gate.human_override("T-LOGIN", user_id="", reason="")
            raise AssertionError("empty override")
        except ValueError:
            pass
        over = gate.human_override("T-LOGIN", user_id="reviewer.li", reason="staging 已人工确认")
        assert over.passed and over.overridden_by == "reviewer.li"
        gate.claim_done("T-LOGIN")
        print("human override with signature ok")

        print("TOY DEMO OK")
    finally:
        shutil.rmtree(root, ignore_errors=True)


demo_verification_gate()


## 3. 生产级：LangChain 门禁工具 + DeepSeek

工具：`run_verification_gate` / `claim_done` / `human_override`。agent 身份不能覆盖。需 `DEEPSEEK_API_KEY`。


In [ ]:
import json
import os
import shutil
import sys
import tempfile
from pathlib import Path
from typing import Any

from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, BaseMessage, HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

sys.path.append(str(Path("../../00_Common").resolve()))
from user_tools import load_project_env  # noqa: E402

load_project_env()

MODEL = "deepseek:deepseek-v4-flash"
PROD_ROOT: Path | None = None
PROD_GATE: VerificationGate | None = None


def get_llm(*, temperature: float = 0.0) -> Any:
    """
    Returns:
        llm: DeepSeek chat model。
    """
    if not os.getenv("DEEPSEEK_API_KEY"):
        raise RuntimeError("DEEPSEEK_API_KEY missing; copy .env.example → .env")
    return init_chat_model(
        MODEL,
        temperature=temperature,
        extra_body={"thinking": {"type": "disabled"}},
    )


def reset_prod_gate(*, scenario: str = "green") -> str:
    """
    Args:
        scenario: green / scope_fail / test_fail / null_exit。
    Returns:
        json: root 与场景。
    """
    global PROD_ROOT, PROD_GATE
    if PROD_ROOT and PROD_ROOT.exists():
        shutil.rmtree(PROD_ROOT, ignore_errors=True)
    PROD_ROOT = Path(tempfile.mkdtemp(prefix="verify_prod_"))
    kwargs: dict[str, Any] = {}
    if scenario == "scope_fail":
        kwargs["scope_ok"] = False
    elif scenario == "test_fail":
        kwargs["acceptance_ok"] = False
    elif scenario == "null_exit":
        kwargs["null_exit"] = True
    write_artifacts(PROD_ROOT, **kwargs)
    PROD_GATE = VerificationGate(PROD_ROOT)
    return json.dumps({"root": str(PROD_ROOT), "scenario": scenario}, ensure_ascii=False)


def _gate() -> VerificationGate:
    if PROD_GATE is None:
        reset_prod_gate(scenario="green")
    assert PROD_GATE is not None
    return PROD_GATE


class EmptyArgs(BaseModel):
    pass


class ScenarioArgs(BaseModel):
    scenario: str = Field("green", description="green / scope_fail / test_fail / null_exit")


class TaskArgs(BaseModel):
    task_id: str = Field("T-LOGIN")


class OverrideArgs(BaseModel):
    task_id: str = Field("T-LOGIN")
    user_id: str = Field(description="人类审查者 id，不能是 agent")
    reason: str = Field(description="记录在案的覆盖原因")


def build_gate_tools() -> list[StructuredTool]:
    def _reset(**kwargs: Any) -> str:
        return reset_prod_gate(scenario=ScenarioArgs(**kwargs).scenario)

    def _run(**kwargs: Any) -> str:
        tid = TaskArgs(**kwargs).task_id
        report = _gate().run(tid)
        return json.dumps(report.to_dict(), ensure_ascii=False)

    def _read(**kwargs: Any) -> str:
        tid = TaskArgs(**kwargs).task_id
        p = _gate().report_path(tid)
        if not p.exists():
            return json.dumps({"ok": False, "error": "report missing"}, ensure_ascii=False)
        return p.read_text(encoding="utf-8")

    def _claim(**kwargs: Any) -> str:
        tid = TaskArgs(**kwargs).task_id
        try:
            report = _gate().claim_done(tid)
            return json.dumps({"ok": True, "done": True, "report": report.to_dict()}, ensure_ascii=False)
        except DoneRefused as e:
            return json.dumps({"ok": False, "done": False, "error": str(e)}, ensure_ascii=False)

    def _override(**kwargs: Any) -> str:
        a = OverrideArgs(**kwargs)
        if a.user_id.lower() in {"agent", "assistant", "model"}:
            return json.dumps({"ok": False, "error": "agent cannot override"}, ensure_ascii=False)
        try:
            report = _gate().human_override(a.task_id, user_id=a.user_id, reason=a.reason)
            return json.dumps({"ok": True, "report": report.to_dict()}, ensure_ascii=False)
        except ValueError as e:
            return json.dumps({"ok": False, "error": str(e)}, ensure_ascii=False)

    return [
        StructuredTool.from_function(
            name="reset_artifacts",
            description="按场景重建物料（green/scope_fail/test_fail/null_exit）。",
            func=_reset,
            args_schema=ScenarioArgs,
        ),
        StructuredTool.from_function(
            name="run_verification_gate",
            description="确定性读取物料并写入 outputs/verification/<task_id>.json。",
            func=_run,
            args_schema=TaskArgs,
        ),
        StructuredTool.from_function(
            name="read_verification_report",
            description="读取固定路径上的验证报告。",
            func=_read,
            args_schema=TaskArgs,
        ),
        StructuredTool.from_function(
            name="claim_done",
            description="仅当门禁通过或已有人类覆盖时才能标完成。",
            func=_claim,
            args_schema=TaskArgs,
        ),
        StructuredTool.from_function(
            name="human_override",
            description="人类覆盖 block；必须提供 user_id 与 reason。agent 身份会被拒。",
            func=_override,
            args_schema=OverrideArgs,
        ),
    ]


GATE_TOOLS = build_gate_tools()


def build_gate_agent():
    """
    Returns:
        agent: 只能跑门禁、不能自行宣布完成的 control agent。
    """
    system = (
        "你是验证门禁控制代理。完成与否只看 run_verification_gate 的报告，"
        "不要用口头判断代替门禁。claim_done 失败时不要假装成功。\n"
        "human_override 只能由人类调用；你不要用自己的名字覆盖。\n"
        "用中文简短说明。"
    )
    return create_agent(get_llm(), GATE_TOOLS, system_prompt=system)


def _print_trace(messages: list[BaseMessage], limit: int = 28) -> None:
    n = 0
    for m in messages:
        if n >= limit:
            break
        if isinstance(m, HumanMessage):
            print(f"USER: {m.content}")
            n += 1
        elif isinstance(m, AIMessage):
            if m.tool_calls:
                for tc in m.tool_calls:
                    print(f"ACTION: {tc['name']}({tc.get('args', {})})")
                    n += 1
            elif m.content:
                print(f"ASSISTANT: {str(m.content)[:280]}")
                n += 1
        elif isinstance(m, ToolMessage):
            print(f"OBS[{m.name}]: {str(m.content)[:240]}")
            n += 1


print(f"verification gate production ready | {MODEL}")


## 4. 生产示例

无 `DEEPSEEK_API_KEY` 则跳过 LLM；脚本路径仍验证绿/红与拒完成。


In [ ]:
def demo_prod_gate() -> None:
    """脚本化门禁 +（有 key 时）DeepSeek。"""
    reset_prod_gate(scenario="green")
    green = _gate().run("T-LOGIN")
    assert green.passed
    _gate().claim_done("T-LOGIN")

    reset_prod_gate(scenario="test_fail")
    bad = _gate().run("T-LOGIN")
    assert not bad.passed
    try:
        _gate().claim_done("T-LOGIN")
        raise AssertionError("must refuse")
    except DoneRefused:
        pass
    print("=== scripted ===")
    print(json.dumps({"green": True, "test_fail_refused": True, "path": str(_gate().report_path("T-LOGIN"))}, ensure_ascii=False))

    if not os.getenv("DEEPSEEK_API_KEY"):
        print("SKIP llm agent: DEEPSEEK_API_KEY missing")
        print("PROD DEMO OK (scripted only)")
        return

    agent = build_gate_agent()
    prompt = (
        "reset_artifacts scenario=test_fail，run_verification_gate task_id=T-LOGIN，"
        "read_verification_report，尝试 claim_done（应失败），"
        "不要自己 human_override；"
        "再 reset_artifacts scenario=green，run_verification_gate，claim_done，"
        "中文说明为何门禁不能含 LLM 判断。"
    )
    result = agent.invoke({"messages": [HumanMessage(content=prompt)]})
    print("=== control agent ===")
    _print_trace(list(result["messages"]))

    final = json.loads(_gate().report_path("T-LOGIN").read_text(encoding="utf-8"))
    assert final.get("passed") is True
    assert final.get("overridden_by") in (None, "")
    print("PROD DEMO OK")


demo_prod_gate()
